# Metrics: balanced outcomes and game closeness

**Milestone:** "Balanced outcomes: define the metric (for example, the share of games drawn or
decided narrowly) and a target over a fixed number of games. Initial aspiration: 95 to 99 percent"
(README Teaching). This notebook makes the definition concrete so it can be argued with.

**Proposed definition** (all thresholds in game.yaml `teaching`):

* A game is **balanced** if it is a draw, or it is decisive **narrowly**: the engine eval (White
  POV, after each ply) stayed within `+-narrow_decisive_cp` for every ply except the last
  `late_plies` (default 10), i.e. nobody was clearly winning until the very end.
* **Balanced share** over N games, with a Wilson 95% interval (with N = 20 and 19 balanced
  games, the interval is roughly 0.76..0.99: say so when reporting).
* Supporting statistics: eval swing (max - min), mean |delta eval| per ply, number of lead changes,
  and a per-game closeness in 0..1.

**TODO checklist**
- [x] Metric functions + Wilson interval.
- [ ] Decide `late_plies` and `narrow_decisive_cp` with real games; maybe use win probability
      instead of centipawns.
- [ ] Fix N (number of games) and the target before user testing (README).

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import yaml

with open(Path("../../gambit_bringup/config/game.yaml")) as _f:
    METRIC_CFG = yaml.safe_load(_f)["teaching"]

## Wilson score interval (for any share k/n)

In [ ]:
def wilson_interval(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    if n == 0:
        return 0.0, 1.0
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)

## Per-game metrics

In [ ]:
@dataclass
class GameOutcome:
    result: str                                   # "1-0" | "0-1" | "1/2-1/2"
    evals_cp: list = field(default_factory=list)  # White POV after each ply
    robot_color: str = "black"
    game_id: str = ""


def is_draw(g: GameOutcome) -> bool:
    return g.result == "1/2-1/2"


def is_narrow_decisive(g: GameOutcome, narrow_cp: int = METRIC_CFG["narrow_decisive_cp"],
                       late_plies: int = 10) -> bool:
    if g.result not in ("1-0", "0-1"):
        return False
    early = g.evals_cp[:-late_plies] if len(g.evals_cp) > late_plies else []
    return all(abs(e) <= narrow_cp for e in early)


def is_balanced(g: GameOutcome, **kw) -> bool:
    return is_draw(g) or is_narrow_decisive(g, **kw)


def eval_swing_stats(evals_cp: list, lead_cp: int = 50, cap: int = 1000) -> dict:
    e = np.clip(np.asarray(evals_cp, dtype=float), -cap, cap)
    if e.size == 0:
        return {"max_abs": 0.0, "swing": 0.0, "mean_abs_delta": 0.0, "lead_changes": 0}
    signs = [int(np.sign(v)) for v in e if abs(v) >= lead_cp]
    changes = sum(1 for a, b in zip(signs, signs[1:]) if a != b)
    return {"max_abs": float(np.abs(e).max()), "swing": float(e.max() - e.min()),
            "mean_abs_delta": float(np.abs(np.diff(e)).mean()) if e.size > 1 else 0.0,
            "lead_changes": changes}


def closeness(evals_cp: list, band: int = METRIC_CFG["closeness_band_cp"] * 4) -> float:
    """1 = eval always 0, 0 = always beyond +-band. Mean of 1 - min(|e|, band)/band."""
    e = np.abs(np.asarray(evals_cp, dtype=float))
    if e.size == 0:
        return 1.0
    return float(np.mean(1 - np.minimum(e, band) / band))

## Over N games

In [ ]:
def balanced_share(games: list[GameOutcome], **kw) -> dict:
    k = sum(is_balanced(g, **kw) for g in games)
    lo, hi = wilson_interval(k, len(games))
    return {"n": len(games), "balanced": k, "share": k / len(games) if games else float("nan"),
            "ci_low": lo, "ci_high": hi,
            "meets_target": lo >= METRIC_CFG["balanced_target"]}


def summarize_games(games: list[GameOutcome]) -> pd.DataFrame:
    rows = []
    for g in games:
        rows.append({"game_id": g.game_id, "result": g.result, "draw": is_draw(g),
                     "narrow_decisive": is_narrow_decisive(g), "balanced": is_balanced(g),
                     "closeness": closeness(g.evals_cp), **eval_swing_stats(g.evals_cp)})
    return pd.DataFrame(rows)

## Self-check

In [ ]:
lo, hi = wilson_interval(19, 20)
assert 0.74 < lo < 0.78 and 0.99 < hi <= 1.0, (lo, hi)
assert wilson_interval(0, 0) == (0.0, 1.0)
close_win = GameOutcome("1-0", [20, -40, 60, 10, -80] + [100, 300, 600, 900, 1200, 2000, 3000, 5000, 9000, 9999])
blowout = GameOutcome("1-0", [50, 400, 800, 1200] + [1500] * 10)
draw = GameOutcome("1/2-1/2", [0, 500, -500, 0])
assert is_narrow_decisive(close_win) and not is_narrow_decisive(blowout) and is_balanced(draw)
s = balanced_share([close_win, blowout, draw])
assert s["balanced"] == 2 and abs(s["share"] - 2 / 3) < 1e-9 and not s["meets_target"]
st = eval_swing_stats([0, 100, -100, 100])
assert st["lead_changes"] == 2 and st["swing"] == 200 and st["max_abs"] == 100
assert closeness([0, 0]) == 1.0 and closeness([10_000]) == 0.0
assert len(summarize_games([close_win, draw])) == 2
print("metrics self-check passed")